# 04 - End-to-End Evaluation + Ablation

**Run this AFTER notebooks 01, 02, 03.** This notebook depends on the pipeline objects defined in 03 (`retriever`, `generator`, `predict_labels_for_image`, `run_pipeline`, `verify`). Easiest path: open `03_rag_pipeline.ipynb`, run all cells, then **paste the cells below at the bottom of that notebook** OR import the helpers via `%run`.

**What this measures (matches the rubric):**
1. **Label-F1** between classifier predictions and IU ground-truth pseudo-labels.
2. **BLEU-1/4, ROUGE-L** between generated report and IU ground-truth Findings+Impression.
3. **BERTScore** for semantic similarity (more meaningful than BLEU for medical text).
4. **Hallucination rate** + **consistency rate** from the verifier.
5. **Ablation table**: no_rag / rag_definition_only / rag_full / rag_full+verifier_loop.

Number of test cases: configurable. Start with 50 to iterate fast, then run the full ~750 IU test set for the report.

In [9]:
!pip install -q nltk rouge-score bert-score sacrebleu evaluate

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# === TXV-based predictor (self-contained: install + import + load) ===
import importlib, subprocess, sys

try:
    import torchxrayvision as xrv
except ImportError:
    print('Installing torchxrayvision...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchxrayvision'])
    # Force Python to refresh its module list
    importlib.invalidate_caches()
    import torchxrayvision as xrv

from torchvision import transforms as T

txv_model = xrv.models.DenseNet(weights='densenet121-res224-all').to(DEVICE).eval()
TXV_LABELS = txv_model.pathologies
TXV_TO_OURS = {c: c for c in CLASS_NAMES}

txv_preprocess = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Lambda(lambda x: (x * 2 - 1) * 1024),
])

@torch.no_grad()
def predict_labels_txv(image_path: str) -> Dict[str, float]:
    img = Image.open(image_path).convert('RGB')
    x = txv_preprocess(img).unsqueeze(0).to(DEVICE)
    out = torch.sigmoid(txv_model(x))[0].cpu().numpy()
    return {c: float(out[TXV_LABELS.index(TXV_TO_OURS[c])]) for c in CLASS_NAMES}

def predict_with_normal_override(image_path: str, predictor=predict_labels_txv,
                                  abnormal_threshold=0.4) -> Dict[str, float]:
    preds = predictor(image_path)
    top = max(preds.values())
    if top < abnormal_threshold:
        return {c: 0.0 for c in CLASS_NAMES}
    return preds

print('TXV predictor ready. Labels match:', all(c in TXV_LABELS for c in CLASS_NAMES))

Installing torchxrayvision...
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
TXV predictor ready. Labels match: True


In [3]:
!rm -f /content/drive/MyDrive/Data/rag_outputs/eval_checkpoint.json

In [7]:
%run '/content/drive/MyDrive/Colab Notebooks/03_rag_pipeline.ipynb'  # loads retriever, generator, predict_labels_for_image, run_pipeline, verify

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Imports OK
KB size: 48


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: pritamdeka/S-PubMedBert-MS-MARCO
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embed dim: 768


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

...

Loaded local model: Qwen/Qwen2.5-7B-Instruct
Active backend: hf_local
Models: groq= llama-3.3-70b-versatile | gemini= gemini-2.5-flash-lite


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loaded local model: Qwen/Qwen2.5-7B-Instruct
Generator backend: hf_local | 
Loaded classifier bundle. Mean test AUC: 0.7579517975602944
TXV predictor ready. Labels match: True
Image: /content/drive/MyDrive/Data/iu_images/2_IM-0652-1001.dcm.png


Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'temperature', 'do_sample', 'max_new_tokens', 'repetition_penalty', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Top-5 predictions:
  Cardiomegaly           0.634
  Fibrosis               0.623
  Nodule                 0.602
  Atelectasis            0.600
  Infiltration           0.584

--- Generated report ---

### FINDINGS

- **Lungs:** Bilateral hyperinflation with flattened diaphragms, suggestive of cardiomegaly. Coarse reticular opacities noted at the lung bases, suggestive of pulmonary fibrosis. Multiple small nodules are seen bilaterally, ranging from 3 to 8 millimeters in diameter. There is no evidence of consolidation or pneumothorax. Possible atelectasis is noted in the left upper lobe. Bilateral interstitial opacities are present, possibly indicative of mild pulmonary edema. No evidence of emphysema, mass, hernia, effusion, pleural thickening, or pneumonia is observed.

- **Heart:** Cardiomegaly is evident with an estimated cardiac silhouette extending beyond the left cardiac border.

- **Mediastinum:** No significant widening or masses are identified within the mediastinum.

- **Pleu

In [10]:
import os, json, time
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support
from rouge_score import rouge_scorer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import sacrebleu

iu_test = pd.read_csv(os.path.join(OUT_DIR, 'iu_test.csv'))
import ast
iu_test['encoded_labels'] = iu_test['encoded_labels'].apply(ast.literal_eval)
print('IU test rows:', len(iu_test))

N_EVAL = 50  # increase to len(iu_test) for the final paper run
iu_eval = iu_test.sample(n=min(N_EVAL, len(iu_test)), random_state=42).reset_index(drop=True)

IU test rows: 501


## 1. Helper variants for the ablation

In [11]:
def variant_no_rag(preds):
    snippets = []
    report = generator.generate(preds, snippets)
    return {'predicted_labels': preds, 'retrieved_snippets': snippets,
            'report': report, 'verifier': verify(report, preds)}

def variant_def_only(preds):
    snippets = retriever.retrieve(preds)
    snippets = [s for s in snippets if s['type'] == 'definition']
    report = generator.generate(preds, snippets)
    return {'predicted_labels': preds, 'retrieved_snippets': snippets,
            'report': report, 'verifier': verify(report, preds)}

def variant_full(preds):
    return run_pipeline(preds, retriever, generator)

def variant_full_verifier(preds):
    return run_pipeline(preds, retriever, generator, with_verifier_loop=True)

VARIANTS = {
    'no_rag':       variant_no_rag,
    'def_only':     variant_def_only,
    'rag_full':     variant_full,
    # 'rag_full+ver': variant_full_verifier,
}

## 2. Run all variants on the eval set

In [12]:
import time, json, os, re

CKPT_PATH = os.path.join(RAG_OUT, 'eval_checkpoint.json')

if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        ckpt = json.load(f)
    all_results = ckpt['all_results']
    all_preds   = ckpt['all_preds']
    start_i     = len(all_preds)
    print(f'Resuming from index {start_i}')
else:
    all_results = {v: [] for v in VARIANTS}
    all_preds   = []
    start_i     = 0

def safe_call(fn, preds, max_wait=900):
    while True:
        try:
            return fn(preds)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'rate_limit' in msg.lower():
                m = re.search(r'in (\d+)m([\d.]+)s', msg)
                wait = int(m.group(1))*60 + float(m.group(2)) + 5 if m else 60
                wait = min(wait, max_wait)
                print(f'  429 hit, sleeping {wait:.0f}s')
                time.sleep(wait)
            else:
                print(f'  non-429 error: {e}')
                return {'predicted_labels': preds, 'retrieved_snippets': [],
                        'report': '', 'verifier': {'is_consistent': False,
                        'hallucinated': [], 'missed': [], 'mentioned': [],
                        'predicted_confident': []}}

for i in range(start_i, len(iu_eval)):
    row = iu_eval.iloc[i]
    try:
        preds = predict_labels_for_image(row['image_path'])
    except Exception as e:
        print(f'[{i}] image fail: {e}'); continue
    all_preds.append(preds)
    for name, fn in VARIANTS.items():
        all_results[name].append(safe_call(fn, preds))
    if (i+1) % 5 == 0:
        with open(CKPT_PATH, 'w') as f:
            json.dump({'all_results': all_results, 'all_preds': all_preds}, f)
        print(f'  {i+1}/{len(iu_eval)} done (checkpointed)')

with open(CKPT_PATH, 'w') as f:
    json.dump({'all_results': all_results, 'all_preds': all_preds}, f)
print('Generation finished.')

Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `

  5/50 done (checkpointed)


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `

  10/50 done (checkpointed)


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `

...

  40/50 done (checkpointed)


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `

  45/50 done (checkpointed)


Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=150) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `

  50/50 done (checkpointed)
Generation finished.


## 3. Compute metrics for each variant

In [13]:
def label_f1(preds_list, gt_label_lists, threshold=0.4):
    P = np.array([[p[c] for c in CLASS_NAMES] for p in preds_list])
    Y = np.array(gt_label_lists)
    Yhat = (P > threshold).astype(int)
    macro_f1 = f1_score(Y, Yhat, average='macro', zero_division=0)
    micro_f1 = f1_score(Y, Yhat, average='micro', zero_division=0)
    return macro_f1, micro_f1

def text_metrics(generated_reports, ground_truth_texts):
    smoothie = SmoothingFunction().method4
    bleu1_list, bleu4_list, rougeL_list = [], [], []
    rs = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    refs_corp, hyps_corp = [], []
    n_skipped = 0
    for hyp, ref in zip(generated_reports, ground_truth_texts):
        if not hyp or not hyp.strip() or not ref or not ref.strip():
            n_skipped += 1
            continue
        try:
            hyp_tok = nltk.word_tokenize(hyp.lower())
            ref_tok = nltk.word_tokenize(ref.lower())
        except LookupError:
            hyp_tok = hyp.lower().split()
            ref_tok = ref.lower().split()
        bleu1_list.append(corpus_bleu([[ref_tok]], [hyp_tok], weights=(1,0,0,0), smoothing_function=smoothie))
        bleu4_list.append(corpus_bleu([[ref_tok]], [hyp_tok], weights=(0.25,)*4, smoothing_function=smoothie))
        rougeL_list.append(rs.score(ref, hyp)['rougeL'].fmeasure)
        refs_corp.append(ref); hyps_corp.append(hyp)
    corpus_bleu_score = sacrebleu.corpus_bleu(hyps_corp, [refs_corp]).score / 100.0 if hyps_corp else 0.0
    return {
        'n_used':      len(bleu1_list),
        'n_skipped':   n_skipped,
        'BLEU-1':      float(np.mean(bleu1_list))  if bleu1_list  else 0.0,
        'BLEU-4':      float(np.mean(bleu4_list))  if bleu4_list  else 0.0,
        'ROUGE-L':     float(np.mean(rougeL_list)) if rougeL_list else 0.0,
        'BLEU-corpus': float(corpus_bleu_score),
    }

def hallucination_stats(results):
    n = len(results)
    if n == 0: return {}
    cons = sum(1 for r in results if r['verifier'].get('is_consistent'))
    hal  = sum(len(r['verifier'].get('hallucinated', [])) for r in results)
    miss = sum(len(r['verifier'].get('missed', []))      for r in results)
    return {'consistency_rate': cons/n, 'avg_hallucinations': hal/n, 'avg_missed': miss/n}

In [14]:
ground_truth_texts = [(r['findings'] or '') + ' ' + (r['impression'] or '') for _, r in iu_eval.iterrows()]
gt_labels = iu_eval['encoded_labels'].tolist()

macro_f1, micro_f1 = label_f1(all_preds, gt_labels)
print(f'Classifier label-F1 on IU pseudo-labels: macro {macro_f1:.3f}  micro {micro_f1:.3f}')

rows = []
for name, results in all_results.items():
    gen = [r['report'] for r in results]
    tm = text_metrics(gen, ground_truth_texts)
    hs = hallucination_stats(results)
    row = {'variant': name}; row.update(tm); row.update(hs)
    rows.append(row)

summary = pd.DataFrame(rows)
print('\n=== Ablation summary ===')
print(summary.to_string(index=False))
summary.to_csv(os.path.join(RAG_OUT, 'ablation_summary.csv'), index=False)

Classifier label-F1 on IU pseudo-labels: macro 0.124  micro 0.091

=== Ablation summary ===
 variant  n_used  n_skipped   BLEU-1   BLEU-4  ROUGE-L  BLEU-corpus  consistency_rate  avg_hallucinations  avg_missed
  no_rag      50          0 0.131283 0.016948 0.133013     0.013512              0.36                0.96        2.28
def_only      50          0 0.125616 0.014249 0.132284     0.010507              0.36                0.94        2.10
rag_full      50          0 0.127502 0.014118 0.130222     0.010657              0.46                0.68        1.90


## 4.BERTScore — semantic similarity

Bertscore is much more meaningful than BLEU for medical text. Uses microsoft/deberta-xlarge-mnli by default, but smaller models work too.

In [15]:
import bert_score
for name, results in all_results.items():
    cands = [r['report'] for r in results]
    if not any(cands):
        continue
    P, R, F1 = bert_score.score(cands, ground_truth_texts, lang='en', verbose=False, batch_size=8)
    summary.loc[summary['variant']==name, 'BERTScore-F1'] = float(F1.mean())
print(summary.to_string(index=False))
summary.to_csv(os.path.join(RAG_OUT, 'ablation_summary.csv'), index=False)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 variant  n_used  n_skipped   BLEU-1   BLEU-4  ROUGE-L  BLEU-corpus  consistency_rate  avg_hallucinations  avg_missed  BERTScore-F1
  no_rag      50          0 0.131283 0.016948 0.133013     0.013512              0.36                0.96        2.28      0.835278
def_only      50          0 0.125616 0.014249 0.132284     0.010507              0.36                0.94        2.10      0.834026
rag_full      50          0 0.127502 0.014118 0.130222     0.010657              0.46                0.68        1.90      0.831401


## 5. Save full per-example outputs for the report appendix

In [16]:
out_rows = []
for i, row in iu_eval.iterrows():
    if i >= len(all_preds): break
    rec = {
        'image':       row['image_path'],
        'gt_findings': row['findings'],
        'gt_impression': row['impression'],
        'predicted_labels': all_preds[i],
    }
    for v in VARIANTS:
        rec[f'report_{v}']    = all_results[v][i]['report']
        rec[f'verifier_{v}']  = all_results[v][i]['verifier']
    out_rows.append(rec)
with open(os.path.join(RAG_OUT, 'per_example_outputs.json'), 'w') as f:
    json.dump(out_rows, f, indent=2)
print('Saved per-example outputs to', os.path.join(RAG_OUT, 'per_example_outputs.json'))

Saved per-example outputs to /content/drive/MyDrive/Data/rag_outputs/per_example_outputs.json


## 6. Sanity inspection — print 3 examples

In [17]:
for i in range(min(3, len(out_rows))):
    rec = out_rows[i]
    print('='*80)
    print(rec['image'])
    print('\n[GT Findings]', rec['gt_findings'])
    print('\n[GT Impression]', rec['gt_impression'])
    print('\n[Top preds]', sorted(rec['predicted_labels'].items(), key=lambda x: -x[1])[:5])
    print('\n[rag_full report]\n', rec['report_rag_full'])
    print('\n[verifier]', rec['verifier_rag_full'])

/content/drive/MyDrive/Data/iu_images/2918_IM-1320-1001.dcm.png

[GT Findings] Cardiomediastinal silhouette is normal. Pulmonary vasculature and are normal. No consolidation, pneumothorax or large pleural effusion. Osseous structures and soft tissues are normal.

[GT Impression] No acute cardiopulmonary disease.

[Top preds] [('Infiltration', 0.5397993922233582), ('Nodule', 0.527773916721344), ('Consolidation', 0.42496541142463684), ('Pneumonia', 0.33283892273902893), ('Pneumothorax', 0.3157058656215668)]

[rag_full report]
 ### FINDINGS

- **Lungs:** Patchy infiltrates are noted in the right lower lobe, which are suggestive of possible pneumonia in the appropriate clinical setting. No consolidation identified.
- **Heart:** Cardiomegaly cannot be excluded based on the current view, but no definitive enlargement is seen.
- **Mediastinum:** No significant widening or masses are identified.
- **Pleura:** No evidence of pneumothorax or pleural effusion is observed.
- **Bones:** No fracture

In [18]:
import shutil, os
SNAP = '/content/drive/MyDrive/Data/rag_outputs/snapshot_final'
os.makedirs(SNAP, exist_ok=True)
for f in ['ablation_summary.csv', 'per_example_outputs.json', 'eval_checkpoint.json']:
    src = f'/content/drive/MyDrive/Data/rag_outputs/{f}'
    if os.path.exists(src):
        shutil.copy2(src, SNAP)
print('Snapshot saved to', SNAP)

Snapshot saved to /content/drive/MyDrive/Data/rag_outputs/snapshot_final


In [19]:
# Recompute verifier with negation-aware logic — no regeneration needed
for variant_name, results in all_results.items():
    for r in results:
        r['verifier'] = verify(r['report'], r['predicted_labels'])

rows = []
for name, results in all_results.items():
    gen = [r['report'] for r in results]
    tm = text_metrics(gen, ground_truth_texts)
    hs = hallucination_stats(results)
    row = {'variant': name}; row.update(tm); row.update(hs)
    rows.append(row)

summary_v2 = pd.DataFrame(rows)
print('=== Ablation summary (negation-aware verifier) ===')
print(summary_v2.to_string(index=False))
summary_v2.to_csv(os.path.join(RAG_OUT, 'ablation_summary_v2.csv'), index=False)
print('\nSaved to', os.path.join(RAG_OUT, 'ablation_summary_v2.csv'))

=== Ablation summary (negation-aware verifier) ===
 variant  n_used  n_skipped   BLEU-1   BLEU-4  ROUGE-L  BLEU-corpus  consistency_rate  avg_hallucinations  avg_missed
  no_rag      50          0 0.131283 0.016948 0.133013     0.013512              0.36                0.96        2.28
def_only      50          0 0.125616 0.014249 0.132284     0.010507              0.36                0.94        2.10
rag_full      50          0 0.127502 0.014118 0.130222     0.010657              0.46                0.68        1.90

Saved to /content/drive/MyDrive/Data/rag_outputs/ablation_summary_v2.csv


In [20]:
# ===== LLM-as-judge evaluation =====
import json, re, time, os
from typing import Dict

JUDGE_SYSTEM = """You are an expert radiologist evaluating chest X-ray report generation.

Given the GROUND-TRUTH report and a GENERATED report, score the generated report on four axes from 0 to 5:

1. FACTUAL_ACCURACY — Does the generated report agree with the ground-truth on the conditions present? (5 = perfect agreement, 0 = contradicts ground truth)
2. COMPLETENESS — Does the generated report cover the same key findings as the ground-truth? (5 = covers everything, 0 = misses everything important)
3. HALLUCINATION — Does the generated report mention conditions or findings NOT present in the ground-truth? (5 = no hallucination, 0 = severe fabrication)
4. CLINICAL_FLUENCY — Is the language appropriate for a radiology report? (5 = professional radiology prose, 0 = clearly non-radiological)

Return ONLY a JSON object with this exact schema:
{"factual_accuracy": int, "completeness": int, "hallucination": int, "clinical_fluency": int, "rationale": "one sentence"}

No prose outside the JSON. No code fences."""

def judge_report(gt_text: str, generated_text: str, backend='gemini') -> Dict:
    user = f"GROUND-TRUTH REPORT:\n{gt_text}\n\nGENERATED REPORT:\n{generated_text}\n\nReturn the JSON evaluation."
    if backend == 'gemini' and generator.gemini_client:
        import google.generativeai as genai
        cfg = genai.types.GenerationConfig(temperature=0.0, max_output_tokens=200)
        client = genai.GenerativeModel('gemini-2.5-flash-lite', system_instruction=JUDGE_SYSTEM)
        time.sleep(4.5)  # respect 15 RPM
        try:
            r = client.generate_content(user, generation_config=cfg)
            txt = r.text
        except Exception as e:
            print(f'judge error: {str(e)[:120]}'); return None
    else:
        # Fallback: use the local Qwen pipeline (acknowledge bias in paper)
        out = generator.hf_pipe(
            [{'role':'system','content':JUDGE_SYSTEM},{'role':'user','content':user}],
            max_new_tokens=200, do_sample=False, return_full_text=False,
            pad_token_id=generator.hf_pipe.tokenizer.eos_token_id,
        )
        gen = out[0]['generated_text']
        txt = gen[-1]['content'] if isinstance(gen, list) else str(gen)
    # Extract JSON
    m = re.search(r'\{[^{}]*\}', txt, re.DOTALL)
    if not m: return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

In [21]:
JUDGE_BACKEND = 'local'  # change to 'local' to use Qwen self-judging if Gemini is throttled

judge_results = {v: [] for v in all_results.keys()}
for i, gt in enumerate(ground_truth_texts):
    if (i+1) % 10 == 0:
        print(f'  judged {i+1}/{len(ground_truth_texts)}')
    for variant_name, results in all_results.items():
        if i >= len(results): continue
        report = results[i]['report']
        if not report or not report.strip():
            judge_results[variant_name].append(None); continue
        scores = judge_report(gt, report, backend=JUDGE_BACKEND)
        judge_results[variant_name].append(scores)

# Aggregate
import numpy as np
judge_rows = []
for name, scores in judge_results.items():
    valid = [s for s in scores if s is not None]
    if not valid:
        judge_rows.append({'variant': name, 'n_judged': 0}); continue
    row = {
        'variant': name,
        'n_judged': len(valid),
        'judge_factual_accuracy': float(np.mean([s.get('factual_accuracy', 0) for s in valid])),
        'judge_completeness':     float(np.mean([s.get('completeness', 0)     for s in valid])),
        'judge_no_hallucination': float(np.mean([s.get('hallucination', 0)    for s in valid])),
        'judge_clinical_fluency': float(np.mean([s.get('clinical_fluency', 0) for s in valid])),
    }
    row['judge_overall'] = (row['judge_factual_accuracy'] + row['judge_completeness']
                             + row['judge_no_hallucination'] + row['judge_clinical_fluency']) / 4
    judge_rows.append(row)

judge_df = pd.DataFrame(judge_rows)
print('\n=== LLM-as-judge scores (0-5, higher is better) ===')
print(judge_df.to_string(index=False))
judge_df.to_csv(os.path.join(RAG_OUT, 'judge_scores.csv'), index=False)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set

  judged 10/50


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  judged 20/50


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

...

  judged 40/50


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

  judged 50/50


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== LLM-as-judge scores (0-5, higher is better) ===
 variant  n_judged  judge_factual_accuracy  judge_completeness  judge_no_hallucination  judge_clinical_fluency  judge_overall
  no_rag        50                    1.36                1.78                    3.98                    2.98          2.525
def_only        50                    1.30                1.72                    4.00                    3.06          2.520
rag_full        50                    1.32                1.84                    3.96                    2.94          2.515


In [22]:
# Combine with surface metrics for the headline table in your paper
final = summary_v2.merge(judge_df, on='variant')
print('\n=== Final combined table ===')
print(final.to_string(index=False))
final.to_csv(os.path.join(RAG_OUT, 'final_ablation_table.csv'), index=False)


=== Final combined table ===
 variant  n_used  n_skipped   BLEU-1   BLEU-4  ROUGE-L  BLEU-corpus  consistency_rate  avg_hallucinations  avg_missed  n_judged  judge_factual_accuracy  judge_completeness  judge_no_hallucination  judge_clinical_fluency  judge_overall
  no_rag      50          0 0.131283 0.016948 0.133013     0.013512              0.36                0.96        2.28        50                    1.36                1.78                    3.98                    2.98          2.525
def_only      50          0 0.125616 0.014249 0.132284     0.010507              0.36                0.94        2.10        50                    1.30                1.72                    4.00                    3.06          2.520
rag_full      50          0 0.127502 0.014118 0.130222     0.010657              0.46                0.68        1.90        50                    1.32                1.84                    3.96                    2.94          2.515


In [23]:
# === Bootstrap 95% confidence intervals over existing 50 reports per variant ===
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
import re

N_BOOT = 1000
RNG    = np.random.default_rng(42)
smoothie = SmoothingFunction().method4
rs = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def _safe_tok(s):
    try:    return nltk.word_tokenize(s.lower())
    except: return s.lower().split()

def _row_metrics(reports, gt_texts):
    """Per-example metrics, vectorised for bootstrap resampling."""
    bleu1, bleu4, rougeL, halluc, missed, consistent = [], [], [], [], [], []
    for hyp, ref, ver in zip(reports, gt_texts, verifiers):
        if not hyp or not hyp.strip() or not ref:
            bleu1.append(np.nan); bleu4.append(np.nan); rougeL.append(np.nan)
            halluc.append(np.nan); missed.append(np.nan); consistent.append(np.nan)
            continue
        ht, rt = _safe_tok(hyp), _safe_tok(ref)
        bleu1.append(corpus_bleu([[rt]],[ht],weights=(1,0,0,0),smoothing_function=smoothie))
        bleu4.append(corpus_bleu([[rt]],[ht],weights=(0.25,)*4,smoothing_function=smoothie))
        rougeL.append(rs.score(ref, hyp)['rougeL'].fmeasure)
        halluc.append(len(ver.get('hallucinated', [])))
        missed.append(len(ver.get('missed',       [])))
        consistent.append(1.0 if ver.get('is_consistent') else 0.0)
    return (np.array(bleu1), np.array(bleu4), np.array(rougeL),
            np.array(halluc), np.array(missed), np.array(consistent))

def bootstrap_ci(values, n_boot=N_BOOT, ci=95):
    """Returns (mean, lower, upper) percentile-bootstrap CI."""
    values = values[~np.isnan(values)]
    if len(values) == 0: return (np.nan, np.nan, np.nan)
    means = []
    for _ in range(n_boot):
        sample = RNG.choice(values, size=len(values), replace=True)
        means.append(sample.mean())
    means = np.sort(means)
    lo = np.percentile(means, (100-ci)/2)
    hi = np.percentile(means, 100 - (100-ci)/2)
    return (values.mean(), lo, hi)

# Per-variant: compute all metrics with CI
ci_rows = []
for variant_name, results in all_results.items():
    reports   = [r['report']   for r in results]
    verifiers = [r['verifier'] for r in results]
    b1, b4, rL, hl, ms, cs = _row_metrics(reports, ground_truth_texts)
    row = {'variant': variant_name}
    for name, arr in [('BLEU-1', b1), ('BLEU-4', b4), ('ROUGE-L', rL),
                      ('avg_hallucinations', hl), ('avg_missed', ms),
                      ('consistency_rate', cs)]:
        m, lo, hi = bootstrap_ci(arr)
        row[f'{name}_mean']  = m
        row[f'{name}_ci']    = f'[{lo:.3f}, {hi:.3f}]'
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows)
print('=== Metrics with 95% bootstrap CI (n_boot=1000) ===\n')
# Render in a compact "mean [lo, hi]" format per metric
for name in ['BLEU-1','BLEU-4','ROUGE-L','consistency_rate','avg_hallucinations','avg_missed']:
    print(f'\n--- {name} ---')
    for _, r in ci_df.iterrows():
        print(f"  {r['variant']:12s}  {r[f'{name}_mean']:.3f}  {r[f'{name}_ci']}")
ci_df.to_csv(os.path.join(RAG_OUT, 'metrics_with_ci.csv'), index=False)
print(f'\nSaved to {os.path.join(RAG_OUT, "metrics_with_ci.csv")}')

=== Metrics with 95% bootstrap CI (n_boot=1000) ===


--- BLEU-1 ---
  no_rag        0.131  [0.119, 0.145]
  def_only      0.126  [0.114, 0.138]
  rag_full      0.128  [0.116, 0.140]

--- BLEU-4 ---
  no_rag        0.017  [0.014, 0.020]
  def_only      0.014  [0.012, 0.017]
  rag_full      0.014  [0.012, 0.016]

--- ROUGE-L ---
  no_rag        0.133  [0.123, 0.144]
  def_only      0.132  [0.122, 0.141]
  rag_full      0.130  [0.121, 0.139]

--- consistency_rate ---
  no_rag        0.360  [0.240, 0.500]
  def_only      0.360  [0.220, 0.500]
  rag_full      0.460  [0.320, 0.600]

--- avg_hallucinations ---
  no_rag        0.960  [0.700, 1.240]
  def_only      0.940  [0.680, 1.200]
  rag_full      0.680  [0.500, 0.900]

--- avg_missed ---
  no_rag        2.280  [1.900, 2.660]
  def_only      2.100  [1.740, 2.520]
  rag_full      1.900  [1.560, 2.260]

Saved to /content/drive/MyDrive/Data/rag_outputs/metrics_with_ci.csv


In [24]:
# === Verifier before/after: naive vs negation-aware on the same report ===

# Define the OLD naive verifier (no negation handling) for contrast
def verify_naive(report: str, predicted_labels, threshold=0.4):
    text = report.lower()
    confident = {l for l, p in predicted_labels.items() if p >= threshold}
    mentioned = set()
    for label, pats in SYNONYMS.items():
        if any(re.search(p, text) for p in pats):
            mentioned.add(label)
    return {
        'mentioned':           sorted(mentioned),
        'predicted_confident': sorted(confident),
        'hallucinated':        sorted(mentioned - confident),
        'missed':              sorted(confident - mentioned),
        'is_consistent':       (mentioned - confident) == set(),
    }

# Pick reports where the two verifiers DISAGREE the most (showcases the fix)
def disagreement_score(report, preds):
    naive = verify_naive(report, preds)
    smart = verify(report, preds)  # negation-aware (already loaded)
    return len(set(naive['hallucinated']) - set(smart['hallucinated']))

candidates = []
for i, r in enumerate(all_results['rag_full']):
    rep = r['report']; preds = r['predicted_labels']
    if not rep or not rep.strip():
        continue
    candidates.append((i, disagreement_score(rep, preds), rep, preds))

candidates.sort(key=lambda x: -x[1])
top = candidates[:2]  # the two most striking examples

for rank, (i, n_corrected, rep, preds) in enumerate(top, 1):
    print('='*90)
    print(f'EXAMPLE {rank}  —  negation-aware verifier removed {n_corrected} false-positive hallucinations')
    print('='*90)
    print(f'\nImage: {iu_eval.iloc[i]["image_path"]}')
    print('\nTop-5 predicted labels:')
    for l, p in sorted(preds.items(), key=lambda x: -x[1])[:5]:
        print(f'  {l:22s} {p:.3f}')
    print('\n--- Generated report ---\n')
    print(rep)
    naive = verify_naive(rep, preds)
    smart = verify(rep, preds)
    print('\n--- NAIVE verifier (regex only) ---')
    print(f'  Mentioned:    {naive["mentioned"]}')
    print(f'  Hallucinated: {naive["hallucinated"]}    <-- FALSE POSITIVES from "no X" phrases')
    print(f'  is_consistent: {naive["is_consistent"]}')
    print('\n--- NEGATION-AWARE verifier (our improved version) ---')
    print(f'  Mentioned:    {smart["mentioned"]}')
    print(f'  Hallucinated: {smart["hallucinated"]}')
    print(f'  is_consistent: {smart["is_consistent"]}')
    diff = set(naive['hallucinated']) - set(smart['hallucinated'])
    print(f'\n  CORRECTED (no longer flagged as hallucination): {sorted(diff)}')
    print()

EXAMPLE 1  —  negation-aware verifier removed 5 false-positive hallucinations

Image: /content/drive/MyDrive/Data/iu_images/575_IM-2173-1001.dcm.png

Top-5 predicted labels:
  Pneumonia              0.856
  Nodule                 0.624
  Pleural_Thickening     0.458
  Infiltration           0.423
  Consolidation          0.332

--- Generated report ---

### FINDINGS

- **Lungs:** Bilateral patchy infiltrates are noted, predominantly in the right lower lobe, which are suggestive of pneumonia. No lobar or segmental consolidation is identified. No parapneumonic effusion is seen.
- **Heart:** Cardiac silhouette and contours are within normal limits.
- **Mediastinum:** Mediastinal structures are unremarkable, without widening or mass effect.
- **Pleura:** No pleural effusion or thickening is identified. No evidence of atelectasis or emphysema is present.
- **Bones:** No bony lesions or fractures are identified.
- **Soft Tissues:** Soft tissues appear unremarkable.

### IMPRESSION

Bilateral